# EJERCICIO 3: SELECCION DE VARIABLES MEDIANTE METODOS DE FILTRO

### cargo el ultimo dataset con las variables creadas a partir de los metodos manuales y automaticos

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
import seaborn as sns

df_ej3=pd.read_csv('dataset_ej2.csv')

df_ej3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3429 entries, 0 to 3428
Data columns (total 68 columns):
 #   Column                                                                                                 Non-Null Count  Dtype  
---  ------                                                                                                 --------------  -----  
 0   country                                                                                                3429 non-null   object 
 1   categoria_expectativa_vida                                                                             3429 non-null   object 
 2   gdp_per_capita_current_us$                                                                             3429 non-null   float64
 3   government_integrity                                                                                   3429 non-null   float64
 4   energy_use_kg_of_oil_equivalent_per_capita                                                      

recordar que este dataset rancio no esta escalado y tiene 0 nulos.... pero antes voy a leer las consignas por las dudas

## 3.1: Filtro basado en varianza y correlacion

- **Filtro de varianza casi nula(VarianceThreshold)**: Pongo lo que entendi... basicamente es un metodo no supervisado(no le pasamos la y,trabaja a ciegas). si el 99% de los paises tuvieran el mismo valor en una columna(por ejemplo que todos tengan acceso a la electricidad), esa variable tendria varianza casi cero, no aportaria nada para distinguir un pais. Este filtro la detecta y la manda a mamar.
- **Filtro por alta correlacion**: ataca la multicolinealidad (redundancia), en el ejercicio 2 creamos variables que basicamente pueden ser clones. Pone un limite de correlacion y quita las que lo superan, bah... una de ellas. Deja el dataset mas limpio

In [5]:
#POR VARIANZA
#guiandome nuevamente con los codigos de la teoria
target = 'gdp_per_capita_current_us$'
cols_texto = ['country', 'categoria_expectativa_vida']

#dividimos aislando las numericas
X = df_ej3.drop(columns=cols_texto + [target])
y = df_ej3[target]

#escalo entre 0 y 1, en la teoria dice que se aplica sobre datos escalados
scaler = MinMaxScaler()
X_escalada = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
#si uso el standarsclaer la varianza daria 1, esto entiendo que haria que el VarianceThreshold no sirva.
#como llevamos todo entre 0 y 1 , si hay una columna casi llena de 0, el filtro la detecta 

# Umbral 0.01: Elimina si el 99% de los valores de la columna son idénticos
umbral_var = 0.01
filtro_var = VarianceThreshold(threshold=umbral_var)
filtro_var.fit(X_escalada)

#nos quedamos solo con las columnas que superaron el umbral
cols_sobrevivientes_var = X_escalada.columns[filtro_var.get_support()]
X_var = X_escalada[cols_sobrevivientes_var]

print(f"columnas base (automáticas + manuales): {len(X_escalada.columns)}")
print(f"sobrevivientes al filtro de varianza: {len(X_var.columns)}")

#POR CORRELACION
#para eliminar redundancia o multicolinealidad (lo que hicimos a pata en el 2)
matriz_corr = X_var.corr().abs()
triangulo_sup = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))

#elegimos un ubral de 0.85 como esta en la teoria
umbral_corr = 0.85
columnas_a_borrar_corr = [col for col in triangulo_sup.columns if any(triangulo_sup[col] > umbral_corr)]
#eliminamos redundancia
X_final_3_1 = X_var.drop(columns=columnas_a_borrar_corr)
print(f"columnas eliminadas por alta correlación (>0.85): {len(columnas_a_borrar_corr)}")
print(f"columnas Sobrevivientes: {len(X_final_3_1.columns)}")



columnas base (automáticas + manuales): 65
sobrevivientes al filtro de varianza: 37
columnas eliminadas por alta correlación (>0.85): 26
columnas Sobrevivientes: 11


Lo del traignulo superior tambien estaba en la teoria, tapa la mitad de abajo y la diagonal y obliga a mirar cada para de variables una vez. Lo copie por las dudas pero entiendo que aca es donde se filtra cual de las dos eliminar. Es decir, el triangulo pone NaN la parte inferior de la matriz, luego si tenemos que dos variables tienen corr=0.9, solo se muestra en la parte de arriba, en la de abajo sale NaN. Entonces cuando hacemos el any() , revisa la primer columna de este par y ve el NaN, luego se encuentra que la otra columna tiene 0.9 con la anterior, y la borra, entonces la primer columna se salvo porque estaba primero en la matriz y su correlacion salia abajo. Medio engorroso pero se entiende

Preguntas para cuando nos tomen: 
- Siempre se aplican ambos metodos?
- El orden importa? es decir, primero varianza luego correlacion?


reconstruyo el df

In [6]:
df_ej3_1= pd.concat([df_ej3[cols_texto], X_final_3_1, y], axis=1)

## 3.2 Seleccion univariada supervisada

- **F-test**: evalua la relacion lineal entre cada predictoria y el target
    - alpha=0.05 (nivel de significancia)
    - H0: el coef de correlacion es 0, no hay relacion entre la caracteristica y el PBI (si p-valor > 0.05, no podemos rechazar H0)
    - H1: el coef de correlacion es distinto de 0, hay relacion lineal significativa (p-valor < 0.05)



In [11]:
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
#usamos las variables numericas que sobrevivieron al ejercicio anterior
X_3_1= X_final_3_1

#nos quedamos con las 5 mejores caract
k_mejores=5

#F-Test
selector_f = SelectKBest(score_func=f_regression, k=k_mejores)
selector_f.fit(X_3_1, y)
df_scores_f = pd.DataFrame({
    'Variable': X_3_1.columns,
    'F-Score (relacion lineal)': selector_f.scores_,
    'p-valor': selector_f.pvalues_
}).sort_values(by='F-Score (relacion lineal)', ascending=False)

print("El top 5 segun el F-test es: ")
print(df_scores_f.head(k_mejores).to_string(index=False))
print("\n" + "="*70 + "\n")



El top 5 segun el F-test es: 
                                           Variable  F-Score (relacion lineal)       p-valor
                               government_integrity                5536.124395  0.000000e+00
bin_auto_energy_use_kg_of_oil_equivalent_per_capita                2844.636112  0.000000e+00
         energy_use_kg_of_oil_equivalent_per_capita                2745.586157  0.000000e+00
               life_expectancy_at_birth_total_years                2146.511107  0.000000e+00
                     agriculture_value_added_of_gdp                1271.562890 3.811877e-237




En este caso, como era de esperar, la variable 'govermente_integrity' gano, pq el metodo es similar al de correlacion de pearson, y esta variable era la q mas se correlacionaba.

- **Informacion mutua**: evalua cualquier tipo de informacion (lineal o no lineal). Mide cuanta incertidumbre (entropia) sobre el PBI desaparece cuando conocemos la variable X. Si la informacion mutua es 0, las variables son independientes

In [12]:
selector_mi = SelectKBest(score_func=mutual_info_regression, k=k_mejores)
selector_mi.fit(X_3_1, y)
df_scores_mi = pd.DataFrame({
    'Variable': X_3_1.columns,
    'Mutual Information Score': selector_mi.scores_
}).sort_values(by='Mutual Information Score', ascending=False)

print("Top 5 variables segun el metodo de INFORMACION MUTUA: ")
print(df_scores_mi.head(k_mejores).to_string(index=False))



Top 5 variables segun el metodo de INFORMACION MUTUA: 
                                                          Variable  Mutual Information Score
                                    agriculture_value_added_of_gdp                  0.984635
ratio_auto_agriculture_value_added_of_gdp_div_government_integrity                  0.981179
                        energy_use_kg_of_oil_equivalent_per_capita                  0.967883
                              life_expectancy_at_birth_total_years                  0.862607
               bin_auto_energy_use_kg_of_oil_equivalent_per_capita                  0.730410


### Conclusion

Estas 4 se repiten en relacion al F-tes:
- energy_use_kg_of_oil_equivalent_per_capita
- bin_auto_energy_use...
- life_expectancy_at_birth_total_years
- agriculture_value_added_of_gdp

Que estas queden en ambos metodos demuestra que son los mejores predictores. Tienen mucha relacion con el PBI sin importar como lo mires.

- Lo novedoso: la variable **agriculture_value_added_of_gdp** pego un salto, quedo ultimo en el F-test y primero en el MI, esto entiendo que se debe a que la relacion entre el peso del agro y el PBI es una curva... los paises pobres dependen mucho del agro pero a medida que se modernizan el porcentaje cae, fue correcta elegirla.

In [13]:
#guardamos los df finales con las 5 mejores de cada metodo
X_final_ftest = X_3_1[X_3_1.columns[selector_f.get_support()]]
X_final_mi = X_3_1[X_3_1.columns[selector_mi.get_support()]]

## 3.3 Comparacion y validacion de modelos

Usamos regresion lineal y medimos el R^2. Es similar al ejercicio 6 del tp anterior 

In [15]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
#definimos los subconjuntos que vamos a poner a competir
subconjuntos = {
    "Modelo BASE (Las 11 variables de 3.1)": X_3_1,
    "Modelo F-TEST (Las 5 ganadoras lineales)": X_final_ftest,
    "Modelo INFO MUTUA (Las 5 ganadoras no lineales)": X_final_mi
}

#validacion cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados={}
for nombre, X_subset in subconjuntos.items():
    
    # transformamos,escalamos y dsp entrenamos
    pipeline = Pipeline([
        ('power_transform', PowerTransformer(method='yeo-johnson')),
        ('scaler', StandardScaler()),
        ('modelo', LinearRegression())
    ])
    
    #aplicamos CV.
    scores = cross_val_score(pipeline, X_subset, y, cv=kf, scoring='r2')
    
    #guardamos el promedio de las 5 evaluaciones
    resultados[nombre] = np.mean(scores)

print("Resultados R2:\n")
for nombre, score in sorted(resultados.items(), key=lambda item: item[1], reverse=True):
    print(f"{nombre}: {score:.4f}")

Resultados R2:

Modelo BASE (Las 11 variables de 3.1): 0.7865
Modelo F-TEST (Las 5 ganadoras lineales): 0.6520
Modelo INFO MUTUA (Las 5 ganadoras no lineales): 0.6277


Bueno si hice todo bien, gano el modelo base y con una diferencia de 14 puntacos.
- Creo que en los otros metodos perdieron por la seleccion univariada... tal vez eliminamos variables que por si sola no aportaban nada pero daban como un contexto diferente para valorar el PIB per capita de un pais

Las 11 variables logran explicar casi e 79% de la varianza del PBI. cuak

# EJERCICIO 4: METODOS WRAPPER Y EMBEBIDOS PARA SELECCION DE VARIABLES

## 4.1 Seleccion secuencial hacia adelante y hacia atras

- **Forward Selection (hacia adelante)**:  Empieza con 0 variables. Prueba las 11 por separado, se queda con la que da mejor R2. Luego prueba sumarle una segunda, y se queda con el mejor dúo. Así sucesivamente hasta llegar al cirterio de corte.
- **Backward Selection (hacia atras)**: Empieza con el equipo completo (11 variables). Prueba sacar a una, y si el R2 no empeora casi nada, la echa. Sigue echando a las peores hasta que queden las que queremos segun algun criterio de corte.



Voy a seguir usando regresion, no uso la libreria de la consiga (sklearn.ensamle), se que es para random forest pero si antes hicimos regresion lo voy a seguir comparando con eso para ver si realmente estos metodos son mejores. Manzanas con manzanas

In [17]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

#el q uso para comparar
modelo_juez = Pipeline([
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler()),
    ('modelo', LinearRegression())
])
k_mejores = 5

#Hacia adelante, en la teoria no habia narnia de esto, hay que estudiar bien q hace
modelo_forward = SequentialFeatureSelector(
    estimator=modelo_juez, 
    n_features_to_select=k_mejores, 
    direction='forward',
    scoring='r2',
    cv=5 
)
modelo_forward.fit(X_3_1, y)
cols_forward = X_3_1.columns[modelo_forward.get_support()]

#Hacia atras
modelo_backward = SequentialFeatureSelector(
    estimator=modelo_juez, 
    n_features_to_select=k_mejores, 
    direction='backward',
    scoring='r2',
    cv=5
)
modelo_backward.fit(X_3_1, y)
cols_backward = X_3_1.columns[modelo_backward.get_support()]

#comparamos
score_forward = cross_val_score(modelo_juez, X_3_1[cols_forward], y, cv=5, scoring='r2').mean()
score_backward = cross_val_score(modelo_juez, X_3_1[cols_backward], y, cv=5, scoring='r2').mean()

#RECORDEMOS QUE EL MODELO BASE DABA  0.7865
print("R2 del modelo con las 11 variables del anterior: 0.7865")
print("\nR2 del mejor univariado: 0.65 ")
print(""*15)
print(f"R2 del modelo_forward: {score_forward}")
print("="*15)
print(f"R2 del modelo_backwar: {score_backward}")


R2 del modelo con las 11 variables del anterior: 0.7865

R2 del mejor univariado: 0.65 

R2 del modelo_forward: 0.7428166689345617
R2 del modelo_backwar: 0.7552594596152418


### Conclusion

En este caso Wrapper > Filtros
- Esto pasa porque wrapper no elige los 5 de manera individual como los metodos anteriores, busca la combinacion de los 5 mejores, los que mejor se complementen.
- Nuestro modelo base requeria de 11 caracteristicas, el Backward con 5 caracteristicas solo pierde 3 puntos, es una ganancia genial, es un modelo mucho mas simple y mas rapido

## 4.2 Metodos embebidos con regularizacion

Metodos embebidos: no testean a ciegas como los filtros ni corren mil veces como los wrapper, ahcen la seleccion al mismo tiempo que aprenden.
- Regularización L1 (Lasso): Es un algoritmo que penaliza la complejidad Mientras entrena, si nota que una variable no aporta casi nada, literalmente encoge el peso (coeficiente) de esa variable hasta llegar exactamente a 0.0. Al quedar en cero, la variable queda eliminada del modelo ("esparsidad").
- Elastic Net: Es una mezcla entre Lasso (L1) y Ridge (L2). Lasso suele ser muy agresivo y, si hay dos variables similares, borra una al azar. Elastic Net reparte el castigo de forma mas chill, siendo mejor para datos con multicolinealidad.

In [18]:
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

#escalar para lasso y elastic es el obligatorio, quise hacer con mixmaxscaler pero daba cualquier cosa
#a estos modelos les gusta la media 0 segun un amigo, pero no me acuerdo por que
scaler = StandardScaler()
X_escalada = pd.DataFrame(scaler.fit_transform(X_3_1), columns=X_3_1.columns, index=X_3_1.index)

#LASSO
modelo_lasso = LassoCV(cv=5, random_state=42)
modelo_lasso.fit(X_escalada, y)
#SelectFromModel extrae solo las columnas a las que Lasso NO les puso coeficiente 0.
selector_lasso = SelectFromModel(modelo_lasso, prefit=True)
cols_lasso = X_escalada.columns[selector_lasso.get_support()]

print(f"sobrevivientes a LASSO= ({len(cols_lasso)} de 11 variables):")
for col in cols_lasso:
    print(f"   - {col}")

#ELASTIC NET (L1+L2)
#l1_ratio=0.5 significa que aplica 50% de agresividad (lasso) y 50% de suavidad (Ridge)
modelo_elastic = ElasticNetCV(l1_ratio=0.5, cv=5, random_state=42)
modelo_elastic.fit(X_escalada, y)
selector_elastic = SelectFromModel(modelo_elastic, prefit=True)
cols_elastic = X_escalada.columns[selector_elastic.get_support()]

print(f"sobrevivientes a ELASTIC NET=({len(cols_elastic)} de 11 variables):")
for col in cols_elastic:
    print(f"   - {col}")

#Comparamos con los anteriores
#usamos el modelo_juez con yeo-johnson del 4.1
score_lasso = cross_val_score(modelo_juez, X_3_1[cols_lasso], y, cv=5, scoring='r2').mean()
score_elastic = cross_val_score(modelo_juez, X_3_1[cols_elastic], y, cv=5, scoring='r2').mean()
print("\n" + "="*50)
print("COMPARATIVA FINAL DE R2 ")
print(f"modelo base (11 var completas): 0.7865")
print(f"mejor del Wrapper (5 var):  0.7552")
print("-" * 30)
print(f"R2 de Lasso ({len(cols_lasso)} var):       {score_lasso:.4f}")
print(f"R2 de ElasticNet ({len(cols_elastic)} var):  {score_elastic:.4f}")



sobrevivientes a LASSO= (8 de 11 variables):
   - government_integrity
   - energy_use_kg_of_oil_equivalent_per_capita
   - life_expectancy_at_birth_total_years
   - services_value_added_of_gdp
   - tiene_dato_gini
   - government_integrity infant_mortality_rate_per_1_000_live_births
   - ratio_auto_agriculture_value_added_of_gdp_div_government_integrity
   - ratio_auto_infant_mortality_rate_per_1_000_live_births_div_agriculture_value_added_of_gdp
sobrevivientes a ELASTIC NET=(4 de 11 variables):
   - government_integrity
   - energy_use_kg_of_oil_equivalent_per_capita
   - life_expectancy_at_birth_total_years
   - bin_auto_energy_use_kg_of_oil_equivalent_per_capita

COMPARATIVA FINAL DE R2 
modelo base (11 var completas): 0.7865
mejor del Wrapper (5 var):  0.7552
------------------------------
R2 de Lasso (8 var):       0.7568
R2 de ElasticNet (4 var):  0.6399


- Lasso le gano un poco al wrapper pero con 3 variables mas.... sigue siendo mejor el de backward selection.

- Elastic net quito muchas variables pero su R2 se desplomo, 4 variables son insuficientes

## 4.3 Comparaciones mas pros

Por las dudas lo ahcemos asi cumplo la consigna

In [21]:
from sklearn.model_selection import cross_validate ,GridSearchCV
from sklearn.linear_model import Ridge

equipos_competidores = {
    "1. BASELINE (Todas las 11 variables)": X_3_1,
    "2. FILTRO Univariado F-Test (5 var)": X_final_ftest,
    "3. WRAPPER Backward Selection (5 var)": X_3_1[cols_backward],
    "4. EMBEBIDO Lasso (Selección automática)": X_3_1[cols_lasso]
}
resultados_finales = []

#CV
for nombre, X_sub in equipos_competidores.items():
    # cross_validate nos permite pedir varias metricas al mismo tiempo
    metricas = ['r2', 'neg_mean_squared_error']
    cv_resultados = cross_validate(
        estimator=modelo_juez, #el del 4.1
        X=X_sub, 
        y=y, 
        cv=5, 
        scoring=metricas
    )
    
    #promediamos el R2
    r2_promedio = np.mean(cv_resultados['test_r2'])

    # Scikit-Learn devuelve el MSE en negativo (porque siempre busca maximizar). 
    # Le ponemos un signo "menos" adelante para volverlo positivo.
    mse_promedio = -np.mean(cv_resultados['test_neg_mean_squared_error'])
    
    resultados_finales.append({
        "Método de Selección": nombre,
        "Variables": len(X_sub.columns),
        "R2": round(r2_promedio, 4),
        "MSE (Más cerca de 0 es MEJOR)": round(mse_promedio, 2)
    })

#tabla final
df_marcador = pd.DataFrame(resultados_finales)
print(df_marcador.to_string(index=False))


                     Método de Selección  Variables     R2  MSE (Más cerca de 0 es MEJOR)
    1. BASELINE (Todas las 11 variables)         11 0.7570                    92684480.67
     2. FILTRO Univariado F-Test (5 var)          5 0.6395                   136889991.65
   3. WRAPPER Backward Selection (5 var)          5 0.7553                    92374809.81
4. EMBEBIDO Lasso (Selección automática)          8 0.7568                    91405688.80


- **Baseline (11 variables)**: Se equivoca por 92.6 millones.
- **Wrapper Backward (5 variables)**: Se equivoca por 92.3 millones. El error bajo papaaaaa
- **Lasso (8 variables)**: Se equivoca por 91.4 millones. Bajo mas pero tiene 3 variables mas que el anterior 

el mas top es wrapper Backward

In [23]:
#Usamos Gridsearch para optimizar el ganador
# Armamos un pipeline nuevo usando Ridge (Regresión Lineal con penalización L2)
pipeline_ridge = Pipeline([
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('modelo_ridge', Ridge())
])
# Le decimos que pruebe estos distintos valores de alpha
grilla_parametros = {
    'modelo_ridge__alpha': [0.1, 1.0, 10.0, 100.0]
}

# GridSearchCV hace CV probando todas las configuraciones
optimizador = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid=grilla_parametros,
    cv=5,
    scoring='r2'
)

#entrenamos solo con el subconjunto ganador del Wrapper Backward
optimizador.fit(X_3_1[cols_backward], y)
print(f"Mejor configuración encontrada: {optimizador.best_params_}")
print(f"R2 superoptimizado del Wrapper: {optimizador.best_score_:.4f}")

Mejor configuración encontrada: {'modelo_ridge__alpha': 10.0}
R2 superoptimizado del Wrapper: 0.7580


Bueno el salto de calidad claramente no estuvo al final, si no al princio... se verifica que el 80% esta en el analisis previo, no sirvio de mucho pero bueno.